In [1]:
import sys
print(sys.executable)

import site
print(site.getsitepackages())

/home1/09967/dongahkim0223/miniconda3/envs/flusion/bin/python
['/home1/09967/dongahkim0223/miniconda3/envs/flusion/lib/python3.11/site-packages']


In [2]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)

print(f"Current working directory changed to: {os.getcwd()}")

sys.path.append(os.path.join(PROJECT_DIR, "code"))
sys.path.append(os.path.join(PROJECT_DIR, "code", "Forecasts"))

import loader
import preprocess_and_plot

Current working directory changed to: /home1/09967/dongahkim0223/Spatial_clustering/code


In [3]:
os.chdir("/home1/09967/dongahkim0223/Spatial_clustering")
print(os.getcwd())

/home1/09967/dongahkim0223/Spatial_clustering


In [5]:
# RAC mapping load (county → RAC)
rac_map = pd.read_csv("./data/tx_rac.csv")[["RAC", "County"]]
rac_map.columns = ["RAC", "county"]
rac_map["county"] = rac_map["county"].astype(str)
TX_dshs = pd.read_csv('data/tx_dshs_region.csv')


In [6]:
CLUSTER_METHODS = ["redcap"]
SEASONS = ["2023-24", "2024-25", "2025-26"]

for season in SEASONS:
    for cluster_method in CLUSTER_METHODS:
        print(f"\n{'='*60}")
        print(f"Method: {cluster_method}")
        print(f"{'='*60}")

        for k in range(5, 46, 2):
            print(f"--- Processing for K = {k} ---")

            input_file  = f"./data/cluster_data_season/df_county_{cluster_method}_exclude_{season}_{k}.csv"
            output_file = f"./data/cluster_data_season/df_county_{cluster_method}_exclude_{season}_{k}_all.csv"

            if not os.path.exists(input_file):
                print(f"  Warning: {input_file} not found. Skipping.")
                continue

            dat = pd.read_csv(input_file)

            # --------------------------------------------------
            # 1. Cluster 레벨
            # --------------------------------------------------
            dat["cluster"] = "G_" + dat["cluster"].astype(str)
            df_cluster = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["cluster"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_cluster["geo_level"] = "cluster"

            # --------------------------------------------------
            # 2. State 레벨
            # --------------------------------------------------
            dat["state"] = "TX"
            df_state = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["state"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_state["geo_level"] = "state"

            # --------------------------------------------------
            # 3. County 레벨
            # --------------------------------------------------
            df_county = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["county"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_county["geo_level"] = "county"

            # --------------------------------------------------
            # 4. RAC 레벨 (county → RAC aggregate)
            # --------------------------------------------------
            dat_rac = dat.merge(rac_map, on="county", how="left")

            # RAC 매핑 안 된 county 경고
            n_missing = dat_rac["RAC"].isna().sum()
            if n_missing > 0:
                missing_counties = dat_rac.loc[dat_rac["RAC"].isna(), "county"].unique()
                print(f"  Warning: {n_missing} rows have no RAC mapping. "
                      f"Counties: {missing_counties}")

            dat_rac = dat_rac.dropna(subset=["RAC"])
            dat_rac["RAC"] = dat_rac["RAC"].astype(str)

            df_rac = preprocess_and_plot.cal_inc_by_group(
                dat=dat_rac,
                group_cols=["RAC"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_rac["geo_level"] = "rac"

            # --------------------------------------------------
            # 5. DSHS 레벨 (county → DSHS aggregate)
            # --------------------------------------------------

            dat3 = dat.merge(TX_dshs, left_on="county", right_on="county", how="left")
            df_dshs = preprocess_and_plot.cal_inc_by_group(
                dat=dat3,
                group_cols=["dshs_region"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_dshs["geo_level"] = "dshs"

            # --------------------------------------------------
            # 6. HSA 레벨 (county → HSA aggregate)
            # --------------------------------------------------
            df_hsa = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["hsa_nci_id"],
                date_col="target_end_date",
                flu_col="value_flu",
                all_col="value_all",
                pop_col="population",
                loader=loader
            )
            df_hsa["geo_level"] = "hsa"



            # --------------------------------------------------
            # 7. concat and save
            # --------------------------------------------------
            df_final = pd.concat([df_cluster, df_county, df_hsa, df_rac, df_dshs, df_state], axis=0)  

            df_final.to_csv(output_file, index=False)
            print(f"  Saved: {output_file}  "
                  f"(rows: cluster={len(df_cluster)}, county={len(df_county)}, "
                  f"state={len(df_state)}, rac={len(df_rac)})")

print("\nAll datasets have been generated successfully!")


Method: redcap
--- Processing for K = 5 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_5_all.csv  (rows: cluster=1000, county=50800, state=200, rac=4400)
--- Processing for K = 7 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_7_all.csv  (rows: cluster=1400, county=50800, state=200, rac=4400)
--- Processing for K = 9 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_9_all.csv  (rows: cluster=1800, county=50800, state=200, rac=4400)
--- Processing for K = 11 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_11_all.csv  (rows: cluster=2200, county=50800, state=200, rac=4400)
--- Processing for K = 13 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_13_all.csv  (rows: cluster=2600, county=50800, state=200, rac=4400)
--- Processing for K = 15 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2023-24_15_all.csv  (rows: cluster=3000, county=50800, state=200

  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_15_all.csv  (rows: cluster=3000, county=50800, state=200, rac=4400)
--- Processing for K = 17 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_17_all.csv  (rows: cluster=3400, county=50800, state=200, rac=4400)
--- Processing for K = 19 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_19_all.csv  (rows: cluster=3800, county=50800, state=200, rac=4400)
--- Processing for K = 21 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_21_all.csv  (rows: cluster=4200, county=50800, state=200, rac=4400)
--- Processing for K = 23 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_23_all.csv  (rows: cluster=4600, county=50800, state=200, rac=4400)
--- Processing for K = 25 ---
  Saved: ./data/cluster_data_season/df_county_redcap_exclude_2025-26_25_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 27 --

In [21]:
CLUSTER_METHODS = ["clustergeo", "skater", "redcap"]
SEASONS = ["2023-24", "2024-25", "2025-26"]

for season in SEASONS:
    for cluster_method in CLUSTER_METHODS:
        print(f"\n{'='*60}")
        print(f"Method: {cluster_method}")
        print(f"{'='*60}")

        for k in range(2, 23):
            print(f"--- Processing for K = {k} ---")

            input_file  = f"./data/cluster_data_season2/df_hsa_{cluster_method}_exclude_{season}_{k}.csv"
            output_file = f"./data/cluster_data_season2/df_hsa_{cluster_method}_exclude_{season}_{k}_all.csv"

            if not os.path.exists(input_file):
                print(f"  Warning: {input_file} not found. Skipping.")
                continue

            dat = pd.read_csv(input_file)

            # --------------------------------------------------
            # 1. Cluster 레벨
            # --------------------------------------------------
            dat["cluster"] = "G_" + dat["cluster"].astype(str)
            df_cleaned = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["cluster"],
                date_col="target_end_date",
                flu_col="hsa_value_flu",
                all_col="hsa_value_all",
                pop_col="hsa_population",
                loader=loader
            )
            df_cleaned["geo_level"] = "cluster" 


            # --------------------------------------------------
            # 2. State 레벨
            # --------------------------------------------------
            dat["state"] = "TX"
            df_state = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["state"],
                date_col="target_end_date",
                flu_col="hsa_value_flu",
                all_col="hsa_value_all",
                pop_col="hsa_population",
                loader=loader
            )
            df_state["geo_level"] = "state" 
    
            # --------------------------------------------------
            # 3. HSA 레벨
            # --------------------------------------------------
            df_hsa = preprocess_and_plot.cal_inc_by_group(
                dat=dat,
                group_cols=["hsa_nci_id"],
                date_col="target_end_date",
                flu_col="hsa_value_flu",
                all_col="hsa_value_all",
                pop_col="hsa_population",
                loader=loader
            )
            df_hsa["geo_level"] = "hsa" 

            # --------------------------------------------------
            # 4. concate and save
            # --------------------------------------------------
            df_final = pd.concat([df_cleaned, df_hsa, df_state], axis=0)
            df_final.to_csv(output_file, index=False)
            print(f"  Saved: {output_file}  "
                  f"(rows: cluster={len(df_cluster)}, county={len(df_county)}, "
                  f"state={len(df_state)}, rac={len(df_rac)})")

print("\nAll datasets have been generated successfully!")


Method: clustergeo
--- Processing for K = 2 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_2_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 3 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_3_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 4 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_4_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 5 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_5_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 6 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_6_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 7 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2023-24_7_all.csv  (rows: cluster=5000, county=50800, state

  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_7_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 8 ---
  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_8_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 9 ---
  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_9_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 10 ---
  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_10_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 11 ---
  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_11_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 12 ---
  Saved: ./data/cluster_data_season/df_hsa_redcap_exclude_2023-24_12_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 13 ---
  Saved: ./data/clust

  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_13_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 14 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_14_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 15 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_15_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 16 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_16_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 17 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_17_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 18 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2024-25_18_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 19 ---
  Saved: ./data/

  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2025-26_20_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 21 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2025-26_21_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 22 ---
  Saved: ./data/cluster_data_season/df_hsa_clustergeo_exclude_2025-26_22_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)

Method: skater
--- Processing for K = 2 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2025-26_2_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 3 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2025-26_3_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 4 ---
  Saved: ./data/cluster_data_season/df_hsa_skater_exclude_2025-26_4_all.csv  (rows: cluster=5000, county=50800, state=200, rac=4400)
--- Processing for K = 5

In [8]:
CLUSTER_METHODS = ["clustergeo", "skater", "redcap"]


for cluster_method in CLUSTER_METHODS:
    print(f"\n{'='*60}")
    print(f"Method: {cluster_method}")
    print(f"{'='*60}")

    for k in range(5, 26):
        print(f"--- Processing for K = {k} ---")

        input_file  = f"./data/cluster_data/df_county_{cluster_method}_{k}.csv"
        output_file = f"./data/cluster_data/df_county_{cluster_method}_{k}_all.csv"

        if not os.path.exists(input_file):
            print(f"  Warning: {input_file} not found. Skipping.")
            continue

        dat = pd.read_csv(input_file)

        # --------------------------------------------------
        # 1. Cluster 레벨
        # --------------------------------------------------
        dat["cluster"] = "G_" + dat["cluster"].astype(str)
        df_cluster = preprocess_and_plot.cal_inc_by_group(
            dat=dat,
            group_cols=["cluster"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_cluster["geo_level"] = "cluster"

        # --------------------------------------------------
        # 2. State 레벨
        # --------------------------------------------------
        dat["state"] = "TX"
        df_state = preprocess_and_plot.cal_inc_by_group(
            dat=dat,
            group_cols=["state"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_state["geo_level"] = "state"

        # --------------------------------------------------
        # 3. County 레벨
        # --------------------------------------------------
        df_county = preprocess_and_plot.cal_inc_by_group(
            dat=dat,
            group_cols=["county"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_county["geo_level"] = "county"

        # --------------------------------------------------
        # 4. RAC 레벨 (county → RAC aggregate)
        # --------------------------------------------------
        dat_rac = dat.merge(rac_map, on="county", how="left")

        # RAC 매핑 안 된 county 경고
        n_missing = dat_rac["RAC"].isna().sum()
        if n_missing > 0:
            missing_counties = dat_rac.loc[dat_rac["RAC"].isna(), "county"].unique()
            print(f"  Warning: {n_missing} rows have no RAC mapping. "
                  f"Counties: {missing_counties}")

        dat_rac = dat_rac.dropna(subset=["RAC"])
        dat_rac["RAC"] = dat_rac["RAC"].astype(str)

        df_rac = preprocess_and_plot.cal_inc_by_group(
            dat=dat_rac,
            group_cols=["RAC"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_rac["geo_level"] = "rac"

        # --------------------------------------------------
        # 5. DSHS 레벨 (county → DSHS aggregate)
        # --------------------------------------------------

        dat3 = dat.merge(TX_dshs, left_on="county", right_on="county", how="left")
        df_dshs = preprocess_and_plot.cal_inc_by_group(
            dat=dat3,
            group_cols=["dshs_region"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_dshs["geo_level"] = "dshs"

        # --------------------------------------------------
        # 6. HSA 레벨 (county → HSA aggregate)
        # --------------------------------------------------
        df_hsa = preprocess_and_plot.cal_inc_by_group(
            dat=dat,
            group_cols=["hsa_nci_id"],
            date_col="target_end_date",
            flu_col="value_flu",
            all_col="value_all",
            pop_col="population",
            loader=loader
        )
        df_hsa["geo_level"] = "hsa"



        # --------------------------------------------------
        # 7. concat and save
        # --------------------------------------------------
        df_final = pd.concat([df_cluster, df_county, df_hsa, df_rac, df_dshs, df_state], axis=0)  

        df_final.to_csv(output_file, index=False)
        print(f"  Saved: {output_file}  "
              f"(rows: cluster={len(df_cluster)}, county={len(df_county)}, "
              f"state={len(df_state)}, rac={len(df_rac)})")

print("\nAll datasets have been generated successfully!")


Method: clustergeo
--- Processing for K = 5 ---
  Saved: ./data/cluster_data/df_county_clustergeo_5_all.csv  (rows: cluster=1160, county=58928, state=232, rac=5104)
--- Processing for K = 6 ---
  Saved: ./data/cluster_data/df_county_clustergeo_6_all.csv  (rows: cluster=1392, county=58928, state=232, rac=5104)
--- Processing for K = 7 ---
  Saved: ./data/cluster_data/df_county_clustergeo_7_all.csv  (rows: cluster=1624, county=58928, state=232, rac=5104)
--- Processing for K = 8 ---
  Saved: ./data/cluster_data/df_county_clustergeo_8_all.csv  (rows: cluster=1856, county=58928, state=232, rac=5104)
--- Processing for K = 9 ---
  Saved: ./data/cluster_data/df_county_clustergeo_9_all.csv  (rows: cluster=2088, county=58928, state=232, rac=5104)
--- Processing for K = 10 ---
  Saved: ./data/cluster_data/df_county_clustergeo_10_all.csv  (rows: cluster=2320, county=58928, state=232, rac=5104)
--- Processing for K = 11 ---
  Saved: ./data/cluster_data/df_county_clustergeo_11_all.csv  (rows: clu

  Saved: ./data/cluster_data/df_county_redcap_17_all.csv  (rows: cluster=3944, county=58928, state=232, rac=5104)
--- Processing for K = 18 ---
  Saved: ./data/cluster_data/df_county_redcap_18_all.csv  (rows: cluster=4176, county=58928, state=232, rac=5104)
--- Processing for K = 19 ---
  Saved: ./data/cluster_data/df_county_redcap_19_all.csv  (rows: cluster=4408, county=58928, state=232, rac=5104)
--- Processing for K = 20 ---
  Saved: ./data/cluster_data/df_county_redcap_20_all.csv  (rows: cluster=4640, county=58928, state=232, rac=5104)
--- Processing for K = 21 ---
  Saved: ./data/cluster_data/df_county_redcap_21_all.csv  (rows: cluster=4872, county=58928, state=232, rac=5104)
--- Processing for K = 22 ---
  Saved: ./data/cluster_data/df_county_redcap_22_all.csv  (rows: cluster=5104, county=58928, state=232, rac=5104)
--- Processing for K = 23 ---
  Saved: ./data/cluster_data/df_county_redcap_23_all.csv  (rows: cluster=5336, county=58928, state=232, rac=5104)
--- Processing for K =